# DeepCF 数据探索

本 notebook 用于探索合成消费网络数据的结构特征。

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from deepcf.data.generator import generate_synthetic_data

sns.set_style("whitegrid")
np.random.seed(42)

data = generate_synthetic_data(num_nodes=300, num_features=16, edge_density=0.05, community_k=5, seed=42)
X = data["features"]; A = data["adjacency"]; W = data["weights"]
labels = data["labels"]; positions = data["positions"]

print(f"Nodes: {A.shape[0]}, Edges: {int(A.sum() // 2)}, Features: {X.shape[1]}, Communities: {len(np.unique(labels))}")

## 1. 度分布

In [ ]:
degrees = A.sum(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(degrees, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Degree Distribution"); axes[0].set_xlabel("Degree"); axes[0].set_ylabel("Count")
axes[1].bar(range(len(degrees)), sorted(degrees, reverse=True), width=1)
axes[1].set_title("Degree Rank Plot"); axes[1].set_xlabel("Node Rank"); axes[1].set_ylabel("Degree")
plt.tight_layout(); plt.show()

## 2. 权重分布

In [ ]:
nonzero_w = W[W > 0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(nonzero_w, bins=30, color="coral", edgecolor="white")
axes[0].set_title("Edge Weight Distribution"); axes[0].set_xlabel("Weight"); axes[0].set_ylabel("Count")
axes[1].scatter(degrees, W.sum(axis=1), alpha=0.5)
axes[1].set_title("Degree vs Total Weight"); axes[1].set_xlabel("Degree"); axes[1].set_ylabel("Total Outgoing Weight")
plt.tight_layout(); plt.show()

## 3. 社区结构可视化

In [ ]:
plt.figure(figsize=(10, 8))
for c in range(5):
    mask = labels == c
    plt.scatter(positions[mask, 0], positions[mask, 1], label=f"Community {c}", s=30, alpha=0.7)
edge_list = np.argwhere(np.triu(A > 0))
sample_idx = np.random.choice(len(edge_list), min(500, len(edge_list)), replace=False)
for idx in sample_idx:
    i, j = edge_list[idx]
    plt.plot([positions[i, 0], positions[j, 0]], [positions[i, 1], positions[j, 1]], color="gray", alpha=0.15, linewidth=0.5)
plt.title("Merchant Consumption Network"); plt.xlabel("X coordinate"); plt.ylabel("Y coordinate")
plt.legend(markerscale=2, fontsize=9); plt.show()

## 4. 图统计汇总

In [ ]:
print("=== Network Statistics ===")
print(f"Nodes: {A.shape[0]}, Edges: {int(A.sum() // 2)}")
print(f"Avg degree: {degrees.mean():.2f}, Max degree: {degrees.max():.0f}")
print(f"Graph density: {A.sum() / (A.shape[0] * (A.shape[0] - 1)):.4f}")
print(f"Avg edge weight: {nonzero_w.mean():.3f} +/- {nonzero_w.std():.3f}")